In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

pd.set_option("display.max_columns", 200)

In [2]:
cd C:\Deadpool\Github_files\F1-ENE-datapipeline-project

C:\Deadpool\Github_files\F1-ENE-datapipeline-project


In [3]:
BRONZE_ROOT = Path("data/bronze/laps/season=2024")
SILVER_ROOT = Path("data/silver/fact_laps")

print("bronze root exists:", BRONZE_ROOT.exists())
SILVER_ROOT.mkdir(parents=True, exist_ok=True)


bronze root exists: True


In [4]:
# Columns that we want to KEEP as original
KEEP_COLS = [
    "season",
    "event_round",
    "session_type",
    "Driver",
    "DriverNumber",
    "Team",
    "LapNumber",
    "Stint",
    "Compound",
    "TyreLife",
    "SpeedI1",
    "SpeedST",
    "TrackStatus",
    "LapStartDate",
    "ingestion_ts",
]

# Timedelta columns to convert → seconds
TIMEDELTA_TO_SECONDS = {
    "LapTime": "LapTime_seconds",
    "Sector1Time": "Sector1Time_seconds",
    "Sector2Time": "Sector2Time_seconds",
    "Sector3Time": "Sector3Time_seconds",
}

# Columns to explicitly DROP
DROP_COLS = [
    "PitInTime",
    "PitOutTime",
    "Time",
    "LapStartTime",
    "Sector1SessionTime",
    "Sector2SessionTime",
    "Sector3SessionTime",
    "DeletedReason",
]


In [6]:
def clean_laps_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Convert timedelta columns to seconds
    for src, tgt in TIMEDELTA_TO_SECONDS.items():
        if src in df.columns:
            df[tgt] = df[src].dt.total_seconds()

    # Normalize IsPersonalBest (bool/object inconsistency)
    if "IsPersonalBest" in df.columns:
        df["IsPersonalBest"] = (
            df["IsPersonalBest"]
            .astype(str)
            .str.lower()
            .map({"true": True, "false": False})
            .fillna(False)
        )

    # Drop unwanted columns
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")

    # Select final Silver columns
    final_cols = KEEP_COLS + list(TIMEDELTA_TO_SECONDS.values()) + ["IsPersonalBest"]
    final_cols = [c for c in final_cols if c in df.columns]

    return df[final_cols]


In [7]:
test_path = BRONZE_ROOT / "round=1" / "session=R" / "data.parquet"
df_test = pd.read_parquet(test_path, engine="pyarrow")

df_clean = clean_laps_df(df_test)

df_clean.head()


,season,event_round,session_type,Driver,DriverNumber,Team,LapNumber,Stint,Compound,TyreLife,SpeedI1,SpeedST,TrackStatus,LapStartDate,ingestion_ts,LapTime_seconds,Sector1Time_seconds,Sector2Time_seconds,Sector3Time_seconds,IsPersonalBest
0,2024,1,R,VER,1,Red Bull Racing,1.0,1.0,SOFT,4.0,234.0,251.0,12,2024-03-02 15:03:42.342,2025-12-15 23:39:57.830341,97.284,NaN,41.266,23.616,False
1,2024,1,R,VER,1,Red Bull Racing,2.0,1.0,SOFT,5.0,232.0,287.0,1,2024-03-02 15:05:19.920,2025-12-15 23:39:57.830341,96.296,30.916,41.661,23.719,True
2,2024,1,R,VER,1,Red Bull Racing,3.0,1.0,SOFT,6.0,231.0,290.0,1,2024-03-02 15:06:56.216,2025-12-15 23:39:57.830341,96.753,30.999,41.966,23.788,False
3,2024,1,R,VER,1,Red Bull Racing,4.0,1.0,SOFT,7.0,233.0,NaN,1,2024-03-02 15:08:32.969,2025-12-15 23:39:57.830341,96.647,30.931,41.892,23.824,False
4,2024,1,R,VER,1,Red Bull Racing,5.0,1.0,SOFT,8.0,231.0,289.0,1,2024-03-02 15:10:09.616,2025-12-15 23:39:57.830341,97.173,31.255,42.056,23.862,False


In [9]:
df_clean.dtypes


season                          int64
event_round                     int64
session_type                   object
Driver                         object
DriverNumber                   object
Team                           object
LapNumber                     float64
Stint                         float64
Compound                       object
TyreLife                      float64
SpeedI1                       float64
SpeedST                       float64
TrackStatus                    object
LapStartDate           datetime64[ns]
ingestion_ts           datetime64[us]
LapTime_seconds               float64
Sector1Time_seconds           float64
Sector2Time_seconds           float64
Sector3Time_seconds           float64
IsPersonalBest                   bool
dtype: object

In [10]:
files = list(BRONZE_ROOT.rglob("data.parquet"))
print(f"Found {len(files)} Bronze lap files")

for src in tqdm(files, desc="Writing Silver laps"):
    df = pd.read_parquet(src, engine="pyarrow")
    df_clean = clean_laps_df(df)

    # Preserve partition structure
    relative_path = src.relative_to(BRONZE_ROOT)
    target_path = SILVER_ROOT / relative_path

    target_path.parent.mkdir(parents=True, exist_ok=True)
    df_clean.to_parquet(target_path, index=False, engine="pyarrow")


Found 24 Bronze lap files


Writing Silver laps:  33%|███▎      | 8/24 [00:00<00:01, 11.76it/s]C:\Users\yashw\AppData\Local\Temp\ipykernel_11948\488323220.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
Writing Silver laps:  42%|████▏     | 10/24 [00:00<00:01, 13.18it/s]C:\Users\yashw\AppData\Local\Temp\ipykernel_11948\488323220.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
C:\Users\yashw\AppData\Local\Temp\ipykernel_11948\488323220.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will chang

In [11]:
sample = Path("data/silver/fact_laps/round=24/session=R/data.parquet")
df_final = pd.read_parquet(sample, engine="pyarrow")

print(df_final.shape)
df_final.head()


(1035, 20)


,season,event_round,session_type,Driver,DriverNumber,Team,LapNumber,Stint,Compound,TyreLife,SpeedI1,SpeedST,TrackStatus,LapStartDate,ingestion_ts,LapTime_seconds,Sector1Time_seconds,Sector2Time_seconds,Sector3Time_seconds,IsPersonalBest
0,2024,24,R,VER,1,Red Bull Racing,1.0,1.0,MEDIUM,1.0,290.0,289.0,12,2024-12-08 13:03:35.034,2025-12-15 23:50:20.007146,99.510,NaN,39.931,34.944,False
1,2024,24,R,VER,1,Red Bull Racing,2.0,1.0,MEDIUM,2.0,291.0,320.0,126,2024-12-08 13:05:14.814,2025-12-15 23:50:20.007146,114.938,18.209,54.569,42.160,True
2,2024,24,R,VER,1,Red Bull Racing,3.0,1.0,MEDIUM,3.0,239.0,321.0,671,2024-12-08 13:07:09.752,2025-12-15 23:50:20.007146,98.051,23.196,41.997,32.858,True
3,2024,24,R,VER,1,Red Bull Racing,4.0,1.0,MEDIUM,4.0,286.0,312.0,1,2024-12-08 13:08:47.803,2025-12-15 23:50:20.007146,89.504,18.166,38.187,33.151,True
4,2024,24,R,VER,1,Red Bull Racing,5.0,1.0,MEDIUM,5.0,288.0,319.0,1,2024-12-08 13:10:17.307,2025-12-15 23:50:20.007146,89.813,18.056,37.945,33.812,False
